Download hugging face model

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Define the model repository ID
model_id = "google/flan-t5-small"

# Download and load the tokenizer
print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Download and load the sequence-to-sequence model
print("Downloading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

print("Download complete! Testing model execution:")

# Quick inference test
input_text = "Answer the following question: What is the capital of France?"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs)

# Decode and print the result
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Result: {response}")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 19649.81it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Download complete! Testing model execution:
Result: l'île de l'île


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
# Removed device_map="auto"
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small") 

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 27059.11it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_dir = "./google_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print(f"Model: {type(model).__name__}")
print(f"Parameters: {parameter_count:,}")
print(f"Trainable parameters: {trainable_count:,}")
print(f"Device: {next(model.parameters()).device}")
print(f"Dtype: {next(model.parameters()).dtype}")
print("Configuration:")
for name in ("d_model", "d_ff", "num_layers", "num_decoder_layers", "num_heads", "vocab_size"):
    print(f"  {name}: {getattr(model.config, name, None)}")

print("First parameter tensors:")
for name, parameter in list(model.named_parameters())[:8]:
    print(f"  {name}: {tuple(parameter.shape)}")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 23137.29it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model: T5ForConditionalGeneration
Parameters: 76,961,152
Trainable parameters: 76,961,152
Device: cpu
Dtype: torch.float32
Configuration:
  d_model: 512
  d_ff: 1024
  num_layers: 8
  num_decoder_layers: 8
  num_heads: 6
  vocab_size: 32128
First parameter tensors:
  shared.weight: (32128, 512)
  encoder.block.0.layer.0.SelfAttention.q.weight: (384, 512)
  encoder.block.0.layer.0.SelfAttention.k.weight: (384, 512)
  encoder.block.0.layer.0.SelfAttention.v.weight: (384, 512)
  encoder.block.0.layer.0.SelfAttention.o.weight: (512, 384)
  encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: (32, 6)
  encoder.block.0.layer.0.layer_norm.weight: (512,)
  encoder.block.0.layer.1.DenseReluDense.wi_0.weight: (1024, 512)


Single query prediction using the model

In [12]:
import torch

query = "translate english to french: How are you?"
model.eval()
inputs = tokenizer(query, return_tensors="pt").to(next(model.parameters()).device)

with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=64)

prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Query: {query}")
print(f"Prediction: {prediction}")

Query: translate english to french: How are you?
Prediction: Comment est ?


Model inference : prediction time for single query

In [13]:
import time

inference_device = next(model.parameters()).device
if inference_device.type == "cuda":
    torch.cuda.synchronize(inference_device)

start_time = time.perf_counter()
with torch.inference_mode():
    timed_generated_ids = model.generate(**inputs, max_new_tokens=64)
if inference_device.type == "cuda":
    torch.cuda.synchronize(inference_device)

inference_seconds = time.perf_counter() - start_time
timed_prediction = tokenizer.decode(timed_generated_ids[0], skip_special_tokens=True)
print(f"Prediction: {timed_prediction}")
print(f"Inference time: {inference_seconds:.4f} seconds")

Prediction: Comment est ?
Inference time: 0.0461 seconds


Inference time for sequential processing

In [15]:
import csv
import time
from pathlib import Path

input_csv = Path("flan_t5_1000_test_inputs.csv")
output_csv = Path("flan_t5_1000_predictions.csv")
max_new_tokens = 128
inference_device = next(model.parameters()).device

processing_start = time.perf_counter()
with input_csv.open("r", newline="", encoding="utf-8-sig") as file:
    test_rows = list(csv.DictReader(file))

if not test_rows:
    raise ValueError(f"No input rows found in {input_csv}")

results = []
total_inference_seconds = 0.0

for row_number, row in enumerate(test_rows, start=1):
    encoded_input = tokenizer(row["prompt"], return_tensors="pt").to(inference_device)
    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)

    inference_start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**encoded_input, max_new_tokens=max_new_tokens)
    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    inference_seconds = time.perf_counter() - inference_start
    total_inference_seconds += inference_seconds

    results.append({
        "id": row["id"],
        "task_type": row["task_type"],
        "prompt": row["prompt"],
        "prediction": tokenizer.decode(generated_ids[0], skip_special_tokens=True),
        "inference_seconds": inference_seconds,
    })

    if row_number % 100 == 0:
        print(f"Processed {row_number}/{len(test_rows)} inputs")

with output_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

total_processing_seconds = time.perf_counter() - processing_start
model_throughput = len(results) / total_inference_seconds
end_to_end_throughput = len(results) / total_processing_seconds
print(f"Rows processed: {len(results)}")
print(f"Total model inference time: {total_inference_seconds:.3f} seconds")
print(f"Average inference time per input: {total_inference_seconds / len(results):.4f} seconds")
print(f"Model inference throughput: {model_throughput:.2f} predictions/second")
print(f"Total processing time (including CSV I/O): {total_processing_seconds:.3f} seconds")
print(f"End-to-end throughput: {end_to_end_throughput:.2f} predictions/second")
print(f"Predictions saved to: {output_csv.resolve()}")

Processed 100/1000 inputs
Processed 200/1000 inputs
Processed 300/1000 inputs
Processed 400/1000 inputs
Processed 500/1000 inputs
Processed 600/1000 inputs
Processed 700/1000 inputs
Processed 800/1000 inputs
Processed 900/1000 inputs
Processed 1000/1000 inputs
Rows processed: 1000
Total model inference time: 47.091 seconds
Average inference time per input: 0.0471 seconds
Model inference throughput: 21.24 predictions/second
Total processing time (including CSV I/O): 47.239 seconds
End-to-end throughput: 21.17 predictions/second
Predictions saved to: /Users/minalghatge/Downloads/MLM_lab/flan_t5_1000_predictions.csv


Inference time for batch processing

In [16]:
import csv
import time
from pathlib import Path

input_csv = Path("flan_t5_1000_test_inputs.csv")
batch_output_csv = Path("flan_t5_1000_predictions_batched.csv")
batch_size = 8
max_new_tokens = 128
inference_device = next(model.parameters()).device

batch_processing_start = time.perf_counter()
with input_csv.open("r", newline="", encoding="utf-8-sig") as file:
    batch_test_rows = list(csv.DictReader(file))

if not batch_test_rows:
    raise ValueError(f"No input rows found in {input_csv}")

batched_results = []
batched_inference_seconds = 0.0

for batch_start in range(0, len(batch_test_rows), batch_size):
    row_batch = batch_test_rows[batch_start:batch_start + batch_size]
    prompt_batch = [row["prompt"] for row in row_batch]
    encoded_batch = tokenizer(prompt_batch, return_tensors="pt", padding=True).to(inference_device)

    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    inference_start = time.perf_counter()
    with torch.inference_mode():
        batch_generated_ids = model.generate(**encoded_batch, max_new_tokens=max_new_tokens)
    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    batch_seconds = time.perf_counter() - inference_start
    batched_inference_seconds += batch_seconds

    batch_predictions = tokenizer.batch_decode(batch_generated_ids, skip_special_tokens=True)
    amortized_seconds_per_input = batch_seconds / len(row_batch)
    for row, prediction in zip(row_batch, batch_predictions):
        batched_results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "prompt": row["prompt"],
            "prediction": prediction,
            "batch_inference_seconds": batch_seconds,
            "amortized_inference_seconds_per_input": amortized_seconds_per_input,
        })

    processed_count = min(batch_start + batch_size, len(batch_test_rows))
    if processed_count % 100 == 0 or processed_count == len(batch_test_rows):
        print(f"Processed {processed_count}/{len(batch_test_rows)} inputs")

with batch_output_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=batched_results[0].keys())
    writer.writeheader()
    writer.writerows(batched_results)

batch_total_processing_seconds = time.perf_counter() - batch_processing_start
batch_average_inference_seconds = batched_inference_seconds / len(batched_results)
batch_inference_throughput = len(batched_results) / batched_inference_seconds
batch_end_to_end_throughput = len(batched_results) / batch_total_processing_seconds
print(f"Rows processed: {len(batched_results)}")
print(f"Batch size: {batch_size}")
print(f"Total model inference time: {batched_inference_seconds:.3f} seconds")
print(f"Average inference time per input: {batch_average_inference_seconds:.4f} seconds")
print(f"Batch inference throughput: {batch_inference_throughput:.2f} predictions/second")
print(f"Total processing time (including CSV I/O): {batch_total_processing_seconds:.3f} seconds")
print(f"End-to-end throughput: {batch_end_to_end_throughput:.2f} predictions/second")
print(f"Batched predictions saved to: {batch_output_csv.resolve()}")

Processed 200/1000 inputs
Processed 400/1000 inputs
Processed 600/1000 inputs
Processed 800/1000 inputs
Processed 1000/1000 inputs
Rows processed: 1000
Batch size: 8
Total model inference time: 35.658 seconds
Average inference time per input: 0.0357 seconds
Batch inference throughput: 28.04 predictions/second
Total processing time (including CSV I/O): 35.715 seconds
End-to-end throughput: 28.00 predictions/second
Batched predictions saved to: /Users/minalghatge/Downloads/MLM_lab/flan_t5_1000_predictions_batched.csv


In [17]:
import csv
import time
from pathlib import Path

input_csv = Path("flan_t5_1000_test_inputs.csv")
batch16_output_csv = Path("flan_t5_1000_predictions_batch16.csv")
batch_size = 16
max_new_tokens = 128
inference_device = next(model.parameters()).device

batch16_processing_start = time.perf_counter()
with input_csv.open("r", newline="", encoding="utf-8-sig") as file:
    batch16_test_rows = list(csv.DictReader(file))

if not batch16_test_rows:
    raise ValueError(f"No input rows found in {input_csv}")

batch16_results = []
batch16_inference_seconds = 0.0

for batch_start in range(0, len(batch16_test_rows), batch_size):
    row_batch = batch16_test_rows[batch_start:batch_start + batch_size]
    prompt_batch = [row["prompt"] for row in row_batch]
    encoded_batch = tokenizer(prompt_batch, return_tensors="pt", padding=True).to(inference_device)

    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    inference_start = time.perf_counter()
    with torch.inference_mode():
        batch_generated_ids = model.generate(**encoded_batch, max_new_tokens=max_new_tokens)
    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    batch_seconds = time.perf_counter() - inference_start
    batch16_inference_seconds += batch_seconds

    batch_predictions = tokenizer.batch_decode(batch_generated_ids, skip_special_tokens=True)
    amortized_seconds_per_input = batch_seconds / len(row_batch)
    for row, prediction in zip(row_batch, batch_predictions):
        batch16_results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "prompt": row["prompt"],
            "prediction": prediction,
            "batch_inference_seconds": batch_seconds,
            "amortized_inference_seconds_per_input": amortized_seconds_per_input,
        })

    processed_count = min(batch_start + batch_size, len(batch16_test_rows))
    if processed_count % 100 == 0 or processed_count == len(batch16_test_rows):
        print(f"Processed {processed_count}/{len(batch16_test_rows)} inputs")

with batch16_output_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=batch16_results[0].keys())
    writer.writeheader()
    writer.writerows(batch16_results)

batch16_total_processing_seconds = time.perf_counter() - batch16_processing_start
batch16_average_inference_seconds = batch16_inference_seconds / len(batch16_results)
batch16_inference_throughput = len(batch16_results) / batch16_inference_seconds
batch16_end_to_end_throughput = len(batch16_results) / batch16_total_processing_seconds
print(f"Rows processed: {len(batch16_results)}")
print(f"Batch size: {batch_size}")
print(f"Total model inference time: {batch16_inference_seconds:.3f} seconds")
print(f"Average inference time per input: {batch16_average_inference_seconds:.4f} seconds")
print(f"Batch inference throughput: {batch16_inference_throughput:.2f} predictions/second")
print(f"Total processing time (including CSV I/O): {batch16_total_processing_seconds:.3f} seconds")
print(f"End-to-end throughput: {batch16_end_to_end_throughput:.2f} predictions/second")
print(f"Batched predictions saved to: {batch16_output_csv.resolve()}")

Processed 400/1000 inputs
Processed 800/1000 inputs
Processed 1000/1000 inputs
Rows processed: 1000
Batch size: 16
Total model inference time: 32.472 seconds
Average inference time per input: 0.0325 seconds
Batch inference throughput: 30.80 predictions/second
Total processing time (including CSV I/O): 32.511 seconds
End-to-end throughput: 30.76 predictions/second
Batched predictions saved to: /Users/minalghatge/Downloads/MLM_lab/flan_t5_1000_predictions_batch16.csv


In [18]:
import csv
import time
from pathlib import Path

input_csv = Path("flan_t5_1000_test_inputs.csv")
batch64_output_csv = Path("flan_t5_1000_predictions_batch64.csv")
batch_size = 64
max_new_tokens = 128
inference_device = next(model.parameters()).device

batch64_processing_start = time.perf_counter()
with input_csv.open("r", newline="", encoding="utf-8-sig") as file:
    batch64_test_rows = list(csv.DictReader(file))

if not batch64_test_rows:
    raise ValueError(f"No input rows found in {input_csv}")

batch64_results = []
batch64_inference_seconds = 0.0

for batch_start in range(0, len(batch64_test_rows), batch_size):
    row_batch = batch64_test_rows[batch_start:batch_start + batch_size]
    prompt_batch = [row["prompt"] for row in row_batch]
    encoded_batch = tokenizer(prompt_batch, return_tensors="pt", padding=True).to(inference_device)

    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    inference_start = time.perf_counter()
    with torch.inference_mode():
        batch_generated_ids = model.generate(**encoded_batch, max_new_tokens=max_new_tokens)
    if inference_device.type == "cuda":
        torch.cuda.synchronize(inference_device)
    batch_seconds = time.perf_counter() - inference_start
    batch64_inference_seconds += batch_seconds

    batch_predictions = tokenizer.batch_decode(batch_generated_ids, skip_special_tokens=True)
    amortized_seconds_per_input = batch_seconds / len(row_batch)
    for row, prediction in zip(row_batch, batch_predictions):
        batch64_results.append({
            "id": row["id"],
            "task_type": row["task_type"],
            "prompt": row["prompt"],
            "prediction": prediction,
            "batch_inference_seconds": batch_seconds,
            "amortized_inference_seconds_per_input": amortized_seconds_per_input,
        })

    processed_count = min(batch_start + batch_size, len(batch64_test_rows))
    if processed_count % 100 == 0 or processed_count == len(batch64_test_rows):
        print(f"Processed {processed_count}/{len(batch64_test_rows)} inputs")

with batch64_output_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=batch64_results[0].keys())
    writer.writeheader()
    writer.writerows(batch64_results)

batch64_total_processing_seconds = time.perf_counter() - batch64_processing_start
batch64_average_inference_seconds = batch64_inference_seconds / len(batch64_results)
batch64_inference_throughput = len(batch64_results) / batch64_inference_seconds
batch64_end_to_end_throughput = len(batch64_results) / batch64_total_processing_seconds
print(f"Rows processed: {len(batch64_results)}")
print(f"Batch size: {batch_size}")
print(f"Total model inference time: {batch64_inference_seconds:.3f} seconds")
print(f"Average inference time per input: {batch64_average_inference_seconds:.4f} seconds")
print(f"Batch inference throughput: {batch64_inference_throughput:.2f} predictions/second")
print(f"Total processing time (including CSV I/O): {batch64_total_processing_seconds:.3f} seconds")
print(f"End-to-end throughput: {batch64_end_to_end_throughput:.2f} predictions/second")
print(f"Batched predictions saved to: {batch64_output_csv.resolve()}")

Processed 1000/1000 inputs
Rows processed: 1000
Batch size: 64
Total model inference time: 39.792 seconds
Average inference time per input: 0.0398 seconds
Batch inference throughput: 25.13 predictions/second
Total processing time (including CSV I/O): 39.822 seconds
End-to-end throughput: 25.11 predictions/second
Batched predictions saved to: /Users/minalghatge/Downloads/MLM_lab/flan_t5_1000_predictions_batch64.csv


## Inference Performance Comparison

Results below are from one run over the same 1,000 prompts. Average time for batched runs is the total generation time divided by the number of inputs; throughput is predictions per second during model generation.

| Processing method | Total inference time (s) | Average time per input (s) | Throughput (predictions/s) |
| --- | ---: | ---: | ---: |
| Sequential | 47.091 | 0.0471 | 21.24 |
| Batch size 8 | 35.658 | 0.0357 | 28.04 |
| Batch size 16 | 32.472 | 0.0325 | 30.80 |
| Batch size 64 | 39.792 | 0.0398 | 25.13 |

Batch size 16 was fastest in this run. Results can vary with system load and hardware.

## Why Batch Size 64 Was Slower

In this CPU run, batch size 64 reached **25.13 predictions/second**, versus **30.80 predictions/second** for batch size 16. A likely contributor is padding: prompts in the CSV range from 11 to 29 tokens, and padding to each batch's longest prompt adds an estimated **37.9%** source-token padding at batch size 64, compared with **16.8%** at batch size 16. The attention mask prevents padded tokens from affecting the answer, but the larger padded tensors still consume compute and memory.

Larger batches also require more activation and decoding memory. Text generation is autoregressive, so sequences in a batch are decoded together; shorter answers can incur extra decoding work while longer answers finish. On a CPU, memory-bandwidth and cache pressure can outweigh the benefit of making fewer model calls.

These are plausible contributors, not a proven breakdown of the slowdown. The measurements are from a single run; repeated warmed-up runs and grouping prompts by token length would help confirm the effect.